In [ ]:
# 0. 라이브러리
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score, roc_auc_score,
                             classification_report, confusion_matrix)

RS = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

In [ ]:
# 1. 데이터 불러오기 + 팀 공통 Train/Test 분할 (index 파일 기준, 80:20 층화)
df = pd.read_csv(ROOT / "data/processed/Faults_processed.csv")      # 원본 26개 피처 + target (A400 제외된 상태)
train_idx = pd.read_csv(ROOT / "data/split/train_idx.csv")["index"]
test_idx = pd.read_csv(ROOT / "data/split/test_idx.csv")["index"]

X, y = df.drop(columns="target"), df["target"]
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print("Train", X_train.shape, "| Test", X_test.shape)

In [ ]:
# 2. 파생변수 10개 생성
PLATE_KEYS = ["Length_of_Conveyer", "Steel_Plate_Thickness", "TypeOfSteel_A300"]   # 강판 조건
DISCRETE = ["TypeOfSteel_A300", "Outside_Global_Index"]                             # 이산형 변수 (마지막 열로 배치)

def plate_key(X):
    return X[PLATE_KEYS].astype(str).agg("_".join, axis=1)

def add_features(X, plate_counts):
    out = X.copy()
    width = (X["X_Maximum"] - X["X_Minimum"]).clip(lower=0)
    mx, mn = X["Maximum_of_Luminosity"], X["Minimum_of_Luminosity"]
    # 형상
    out["compactness"] = (X["X_Perimeter"] + X["Y_Perimeter"]) / np.sqrt(X["Pixels_Areas"])   # 둘레 대비 면적
    out["perimeter_ratio_log"] = np.log((X["X_Perimeter"] + 1) / (X["Y_Perimeter"] + 1))      # 가로/세로 둘레 비
    out["x_straightness"] = X["X_Perimeter"] / (width + 1)                                     # 가로 윤곽의 굴곡
    # 밝기
    out["lum_range"] = mx - mn                                                                 # 밝기 범위
    out["lum_contrast"] = (mx - mn) / (mx + mn).replace(0, 1)                                  # 상대 대비
    out["lum_mean_position"] = (X["Sum_of_Luminosity"] / X["Pixels_Areas"] - mn) / (mx - mn + 1)  # 평균 밝기의 위치
    # 위치
    xc = (X["X_Minimum"] + X["X_Maximum"]) / 2 / X["Length_of_Conveyer"]
    out["x_center_rel"] = xc                                                                   # 컨베이어 폭 대비 중심 위치
    out["x_center_dist"] = (xc - 0.5).abs()                                                    # 중앙에서 떨어진 정도
    # 강판 맥락
    out["plate_defect_count"] = plate_key(X).map(plate_counts).fillna(0).astype(float)        # 같은 강판 조건의 결함 수 (처음 보는 조건 = 0)
    out["thickness_per_conveyer"] = X["Steel_Plate_Thickness"] / X["Length_of_Conveyer"]
    # 연속형 → 이산형 순서로 열 정렬 (학습·평가 열 순서 고정)
    return out[[c for c in out.columns if c not in DISCRETE] + DISCRETE]

In [ ]:
# 3. 최종 모델: ExtraTrees + Random Forest + GBM Soft Voting (36개 피처에서 튜닝한 하이퍼파라미터)
def fit_soft_voting(X_tr, y_tr, n_jobs=-1):
    plate_counts = plate_key(X_tr).value_counts().to_dict()      # 강판 조건별 결함 수는 학습 데이터로만 계산 (누수 방지)
    Xf = add_features(X_tr, plate_counts)
    et = ExtraTreesClassifier(n_estimators=438, max_depth=15, max_features=0.7, min_samples_leaf=1,
                              class_weight="balanced", random_state=RS, n_jobs=n_jobs)     # 불균형: 클래스 가중치
    rf = RandomForestClassifier(n_estimators=483, max_depth=None, max_features=0.3, min_samples_leaf=1,
                                random_state=RS, n_jobs=n_jobs)                             # 불균형 처리 없음
    gbm = GradientBoostingClassifier(n_estimators=240, learning_rate=0.08793919399591556, max_depth=4,
                                     subsample=0.9079017872741743, max_features=0.5, min_samples_leaf=5,
                                     random_state=RS)
    # GBM은 class_weight 인자가 없으므로 균형 가중치를 sample_weight로 전달: w_c = 최다 클래스 건수 / 클래스 c 건수
    counts = y_tr.value_counts()
    et.fit(Xf, y_tr); rf.fit(Xf, y_tr); gbm.fit(Xf, y_tr, sample_weight=y_tr.map(counts.max() / counts).to_numpy())
    return [et, rf, gbm], plate_counts

def predict_soft_voting(models, plate_counts, X_new):
    Xf = add_features(X_new, plate_counts)
    proba = np.mean([m.predict_proba(Xf) for m in models], axis=0)   # 세 모델의 클래스별 확률 평균
    pred = models[0].classes_[proba.argmax(axis=1)]                   # 확률이 가장 높은 클래스
    return pred, proba

def metrics(y_true, pred, proba, classes):
    return {"Macro F1": f1_score(y_true, pred, average="macro"),
            "Accuracy": accuracy_score(y_true, pred),
            "Macro Precision": precision_score(y_true, pred, average="macro"),
            "Macro Recall": recall_score(y_true, pred, average="macro"),
            "Weighted F1": f1_score(y_true, pred, average="weighted"),
            "Macro ROC-AUC": roc_auc_score(y_true, proba, multi_class="ovr", average="macro", labels=classes)}

In [ ]:
# 4-1. 팀 공통 기준: Train 내부 StratifiedKFold 5-fold (shuffle=True, random_state=42)
def eval_fold(tr, va):
    # fold마다 학습 fold로만 파생변수(강판 조건별 결함 수)·모델 학습 → 검증 fold 평가
    models, pc = fit_soft_voting(X_train.iloc[tr], y_train.iloc[tr], n_jobs=1)
    pred, proba = predict_soft_voting(models, pc, X_train.iloc[va])
    return metrics(y_train.iloc[va], pred, proba, models[0].classes_)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
cv5_scores = pd.DataFrame(Parallel(n_jobs=-1)(delayed(eval_fold)(tr, va) for tr, va in cv.split(X_train, y_train)))
cv5_scores.index = [f"fold {i + 1}" for i in range(len(cv5_scores))]
print(f"5-fold CV Macro F1 : {cv5_scores['Macro F1'].mean():.4f} ± {cv5_scores['Macro F1'].std(ddof=0):.4f}")
cv5_summary = pd.DataFrame({"평균": cv5_scores.mean(), "표준편차": cv5_scores.std(ddof=0)}).round(4)   # 지표별 5 fold 평균 ± 표준편차
cv5_summary

In [ ]:
# 4-2. 추가 검증: 반복 층화 CV (5-fold × 3회 = 15 fold) — 분할 운의 영향을 줄인 평균
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RS)

cv_scores = pd.DataFrame(Parallel(n_jobs=-1)(delayed(eval_fold)(tr, va) for tr, va in rcv.split(X_train, y_train)))   # 몇 분 소요
cv_scores.index = [f"fold {i + 1}" for i in range(len(cv_scores))]
print(f"반복 CV(5×3) Macro F1 : {cv_scores['Macro F1'].mean():.4f} ± {cv_scores['Macro F1'].std(ddof=0):.4f}"
      f"   (0.83 이상 fold {(cv_scores['Macro F1'] >= 0.83).sum()}/15)")
cv_summary = pd.DataFrame({"평균": cv_scores.mean(), "표준편차": cv_scores.std(ddof=0)}).round(4)   # 지표별 15 fold 평균 ± 표준편차
cv_summary

In [ ]:
# 4-3. 과적합 점검: 같은 5-fold에서 학습 fold 점수 vs 검증 fold 점수 (모델별)
def overfit_fold(tr, va):
    X_tr, y_tr, X_va, y_va = X_train.iloc[tr], y_train.iloc[tr], X_train.iloc[va], y_train.iloc[va]
    models, pc = fit_soft_voting(X_tr, y_tr, n_jobs=1)
    F_tr, F_va = add_features(X_tr, pc), add_features(X_va, pc)
    row = {}
    for name, m in zip(["ExtraTrees", "Random Forest", "GBM"], models):              # 구성 모델 각각
        row[(name, "Train")] = f1_score(y_tr, m.predict(F_tr), average="macro")
        row[(name, "Validation")] = f1_score(y_va, m.predict(F_va), average="macro")
    row[("Soft Voting", "Train")] = f1_score(y_tr, predict_soft_voting(models, pc, X_tr)[0], average="macro")
    row[("Soft Voting", "Validation")] = f1_score(y_va, predict_soft_voting(models, pc, X_va)[0], average="macro")
    return row

of = pd.DataFrame(Parallel(n_jobs=-1)(delayed(overfit_fold)(tr, va) for tr, va in cv.split(X_train, y_train)))
of.columns = pd.MultiIndex.from_tuples(of.columns)                                # (모델, Train/Validation)
overfit = of.mean().unstack()[["Train", "Validation"]]
overfit["Gap (Train − Validation)"] = overfit["Train"] - overfit["Validation"]
overfit["Validation 표준편차"] = of.xs("Validation", axis=1, level=1).std(ddof=0)
overfit.round(4)

In [ ]:
# 4-4. 학습 곡선: 학습 데이터 양을 늘릴 때 Train/Validation 성능(Macro F1)과 손실(Log Loss) 변화
#      (같은 5-fold, 각 학습 fold에서 일부만 층화 추출해 학습 → 같은 검증 fold로 평가)
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:   # 한글 폰트 (Windows: 맑은 고딕)
    if any(font == f.name for f in mpl.font_manager.fontManager.ttflist):
        mpl.rcParams["font.family"] = font; break
mpl.rcParams["axes.unicode_minus"] = False

def lc_fold(tr, va, frac):
    X_tr, y_tr = X_train.iloc[tr], y_train.iloc[tr]
    if frac < 1:   # 학습 fold에서 frac 비율만 클래스 비율을 유지하며 추출
        X_tr, _, y_tr, _ = train_test_split(X_tr, y_tr, train_size=frac, stratify=y_tr, random_state=RS)
    models, pc = fit_soft_voting(X_tr, y_tr, n_jobs=1)
    classes = models[0].classes_
    pred_tr, proba_tr = predict_soft_voting(models, pc, X_tr)
    pred_va, proba_va = predict_soft_voting(models, pc, X_train.iloc[va])
    y_va = y_train.iloc[va]
    return {"학습 비율": frac, "학습 건수": len(y_tr),
            "Train F1": f1_score(y_tr, pred_tr, average="macro"), "Validation F1": f1_score(y_va, pred_va, average="macro"),
            "Train Loss": log_loss(y_tr, proba_tr, labels=classes), "Validation Loss": log_loss(y_va, proba_va, labels=classes)}

FRACS = [0.2, 0.4, 0.6, 0.8, 1.0]
lc = pd.DataFrame(Parallel(n_jobs=-1)(delayed(lc_fold)(tr, va, f) for f in FRACS for tr, va in cv.split(X_train, y_train)))   # 몇 분 소요
lc_mean, lc_std = lc.groupby("학습 비율").mean(), lc.groupby("학습 비율").std(ddof=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, key, title in [(axes[0], "F1", "Macro F1 (높을수록 좋음)"), (axes[1], "Loss", "Log Loss (낮을수록 좋음)")]:
    for part, color in [("Train", "tab:blue"), ("Validation", "tab:orange")]:
        m, s = lc_mean[f"{part} {key}"], lc_std[f"{part} {key}"]
        ax.plot(lc_mean["학습 건수"], m, "o-", color=color, label=part)
        ax.fill_between(lc_mean["학습 건수"], m - s, m + s, color=color, alpha=0.15)   # 5-fold 표준편차
    ax.set_xlabel("학습 데이터 건수"); ax.set_title(title); ax.grid(alpha=0.3); ax.legend()
fig.suptitle("학습 곡선 — 최종 모델 Soft Voting (5-fold 평균 ± 표준편차)")
plt.tight_layout(); plt.show()
lc_mean.round(4)

In [ ]:
# 4-5. 손실 곡선: 트리 수(학습 반복)가 늘어날 때 Train/Validation Log Loss 변화 — 구성 모델별 (같은 5-fold)
#      검증 손실이 어느 시점부터 다시 올라가면 과적합 신호
def loss_curves(tr, va):
    X_tr, y_tr, X_va, y_va = X_train.iloc[tr], y_train.iloc[tr], X_train.iloc[va], y_train.iloc[va]
    models, pc = fit_soft_voting(X_tr, y_tr, n_jobs=1)
    F_tr, F_va = add_features(X_tr, pc), add_features(X_va, pc)
    out = {}
    for name, m in zip(["ExtraTrees", "Random Forest"], models[:2]):          # 배깅: 트리 1개 ~ N개까지 누적 평균 확률
        for part, F, y in [("Train", F_tr, y_tr), ("Validation", F_va, y_va)]:
            P = np.cumsum([t.predict_proba(F.values) for t in m.estimators_], axis=0)   # 개별 트리는 배열 입력
            P /= np.arange(1, len(m.estimators_) + 1)[:, None, None]
            out[(name, part)] = [log_loss(y, p, labels=m.classes_) for p in P]
    gbm = models[2]                                                            # 부스팅: 반복(트리) 단계별 확률
    for part, F, y in [("Train", F_tr, y_tr), ("Validation", F_va, y_va)]:
        out[("GBM", part)] = [log_loss(y, p, labels=gbm.classes_) for p in gbm.staged_predict_proba(F)]
    out[("GBM", "Validation F1")] = [f1_score(y_va, p, average="macro") for p in gbm.staged_predict(F_va)]   # 반복별 검증 Macro F1
    return out

curves = Parallel(n_jobs=-1)(delayed(loss_curves)(tr, va) for tr, va in cv.split(X_train, y_train))   # 몇 분 소요

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
loss_summary = {}
for ax, name in zip(axes, ["ExtraTrees", "Random Forest", "GBM"]):
    tr_arr = np.array([c[(name, "Train")] for c in curves])                   # (fold, 트리 수)
    va_arr = np.array([c[(name, "Validation")] for c in curves])
    x = np.arange(1, tr_arr.shape[1] + 1)
    for arr, part, color in [(tr_arr, "Train", "tab:blue"), (va_arr, "Validation", "tab:orange")]:
        ax.plot(x, arr.mean(0), color=color, label=part)
        ax.fill_between(x, arr.mean(0) - arr.std(0), arr.mean(0) + arr.std(0), color=color, alpha=0.15)
    best = int(va_arr.mean(0).argmin()) + 1                                    # 검증 손실이 가장 낮은 트리 수
    ax.axvline(best, color="gray", ls="--", lw=1)
    ax.set_xlabel("트리 수 (GBM은 부스팅 반복 횟수)"); ax.set_ylabel("Log Loss"); ax.set_title(name)
    ax.set_ylim(0, 2 if name != "GBM" else None); ax.grid(alpha=0.3)   # 배깅은 트리 1~2개일 때 손실이 매우 커서 0~2 구간만 표시
    if name == "GBM":   # GBM은 반복별 검증 Macro F1도 함께 (오른쪽 축)
        f1_arr = np.array([c[("GBM", "Validation F1")] for c in curves])
        ax2 = ax.twinx(); ax2.plot(x, f1_arr.mean(0), color="tab:green", lw=1.2, label="Validation Macro F1 (오른쪽 축)")
        ax2.set_ylabel("Macro F1", color="tab:green")
        h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc="center right")
    else:
        ax.legend(loc="upper right")
    loss_summary[name] = {"트리 수(최종)": len(x), "Train Loss 최종": tr_arr.mean(0)[-1],
                          "Validation Loss 최소": va_arr.mean(0).min(), "최소 지점(트리 수)": best,
                          "Validation Loss 최종": va_arr.mean(0)[-1]}
f1_mean = np.array([c[("GBM", "Validation F1")] for c in curves]).mean(0)
loss_summary["GBM"].update({"검증 F1 @손실 최소": f1_mean[loss_summary["GBM"]["최소 지점(트리 수)"] - 1],
                            "검증 F1 최대": f1_mean.max(), "F1 최대 지점": int(f1_mean.argmax()) + 1, "검증 F1 최종": f1_mean[-1]})
fig.suptitle("손실 곡선 — 구성 모델별 Train/Validation Log Loss (5-fold 평균 ± 표준편차, 점선 = 검증 손실 최소 지점)")
plt.tight_layout(); plt.show()
pd.DataFrame(loss_summary).T.round(4)

In [ ]:
# 4-5b. 최종 Soft Voting 손실 곡선: 세 구성 모델을 같은 비율(학습 진행도)로 함께 키우며 앙상블 Train/Validation Log Loss 계산
#       진행도 100% = ExtraTrees 438개·RF 483개 트리, GBM 240회 반복 (같은 5-fold)
GRID = np.linspace(0.01, 1.0, 100)                                            # 학습 진행도 1% ~ 100%

def sv_loss_curve(tr, va):
    X_tr, y_tr, X_va, y_va = X_train.iloc[tr], y_train.iloc[tr], X_train.iloc[va], y_train.iloc[va]
    models, pc = fit_soft_voting(X_tr, y_tr, n_jobs=1)
    et, rf, gbm = models
    cls = et.classes_
    out = {}
    for part, X_, y_ in [("Train", X_tr, y_tr), ("Validation", X_va, y_va)]:
        F = add_features(X_, pc)
        cum = []                                                              # 모델별 단계 확률: (단계 수, 샘플, 클래스)
        for m in (et, rf):                                                    # 배깅: 트리 1~N개 누적 평균
            P = np.cumsum([t.predict_proba(F.values) for t in m.estimators_], axis=0)
            cum.append(P / np.arange(1, len(m.estimators_) + 1)[:, None, None])
        cum.append(np.array(list(gbm.staged_predict_proba(F))))              # 부스팅: 반복 단계별 확률
        losses, f1s = [], []
        for g in GRID:
            p = np.mean([P[max(1, int(round(g * len(P)))) - 1] for P in cum], axis=0)   # 같은 진행도의 세 모델 확률 평균
            losses.append(log_loss(y_, p, labels=cls))
            f1s.append(f1_score(y_, cls[p.argmax(1)], average="macro"))
        out[part] = (np.array(losses), np.array(f1s))
    return out

sv_curves = Parallel(n_jobs=-1)(delayed(sv_loss_curve)(tr, va) for tr, va in cv.split(X_train, y_train))

x = GRID * 100
fig, ax = plt.subplots(figsize=(9, 4.6))
for part, color in [("Train", "tab:blue"), ("Validation", "tab:orange")]:
    arr = np.array([c[part][0] for c in sv_curves])                           # (fold, 진행도)
    ax.plot(x, arr.mean(0), color=color, label=f"{part} Log Loss")
    ax.fill_between(x, arr.mean(0) - arr.std(0), arr.mean(0) + arr.std(0), color=color, alpha=0.15)
va_loss = np.array([c["Validation"][0] for c in sv_curves]).mean(0)
va_f1 = np.array([c["Validation"][1] for c in sv_curves]).mean(0)
best = int(va_loss.argmin())
ax.axvline(x[best], color="gray", ls="--", lw=1, label=f"검증 손실 최소 ({x[best]:.0f}%)")
ax.set_xlabel("학습 진행도 (%) — 100% = ET 438·RF 483 트리, GBM 240회"); ax.set_ylabel("Log Loss")
ax.set_ylim(0, 2); ax.grid(alpha=0.3)                                         # 진행 초반(트리 1~2개)은 손실이 매우 커서 0~2 구간만 표시
ax2 = ax.twinx(); ax2.plot(x, va_f1, color="tab:green", lw=1.2, label="Validation Macro F1 (오른쪽 축)")
ax2.set_ylabel("Macro F1", color="tab:green")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="center right", fontsize=9)
ax.set_title("최종 Soft Voting 손실 곡선 (5-fold 평균 ± 표준편차)")
plt.tight_layout(); plt.show()

pd.Series({"Validation Loss 최소": va_loss.min(), "최소 지점(진행도 %)": x[best], "Validation Loss 최종(100%)": va_loss[-1],
           "검증 F1 @손실 최소": va_f1[best], "검증 F1 최종(100%)": va_f1[-1]}).round(4)

In [ ]:
# 5. Train 전체로 학습 → Test 예측
models, plate_counts = fit_soft_voting(X_train, y_train)
pred, proba = predict_soft_voting(models, plate_counts, X_test)

In [ ]:
# 6. Test 성능
order = ["Other_Faults", "Bumps", "K_Scratch", "Z_Scratch", "Pastry", "Stains", "Dirtiness"]
test_scores = pd.Series(metrics(y_test, pred, proba, models[0].classes_)).round(4)
print("Test 성능"); print(test_scores.to_string(), "\n")
print(f"CV(5-fold) Macro F1 {cv5_scores['Macro F1'].mean():.4f} vs Test {test_scores['Macro F1']:.4f}  → 차이 {cv5_scores['Macro F1'].mean() - test_scores['Macro F1']:+.4f}\n")   # 과적합 점검: 처음 보는 데이터에서도 CV 수준 유지되는지
print(classification_report(y_test, pred, labels=order, digits=3))
pd.DataFrame(confusion_matrix(y_test, pred, labels=order), index=order, columns=order)   # 행 = 실제, 열 = 예측

In [ ]:
# 6-1. 최종 모델 Confusion Matrix 시각화 + 주요 결과 요약 (Test 389건)
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:   # 한글 폰트
    if any(font == f.name for f in mpl.font_manager.fontManager.ttflist):
        mpl.rcParams["font.family"] = font; break
mpl.rcParams["axes.unicode_minus"] = False

order = ["Other_Faults", "Bumps", "K_Scratch", "Z_Scratch", "Pastry", "Stains", "Dirtiness"]   # 건수 내림차순
cm = confusion_matrix(y_test, pred, labels=order)
pct = cm / cm.sum(axis=1, keepdims=True) * 100                                   # 행(실제) 기준 비율
p, r, f, n = precision_recall_fscore_support(y_test, pred, labels=order)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]                                                                     # (왼쪽) 혼동행렬: 색 = 행 기준 %, 숫자 = % (건수)
ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
for i in range(len(order)):
    for j in range(len(order)):
        if cm[i, j]:
            color = "white" if pct[i, j] > 55 else "black"
            ax.text(j, i - 0.13, f"{pct[i, j]:.1f}%", ha="center", va="center", fontsize=9, color=color, fontweight="bold" if i == j else None)
            ax.text(j, i + 0.22, f"({cm[i, j]})", ha="center", va="center", fontsize=8, color=color)
ax.set_xticks(range(len(order)), order, rotation=35, ha="right")
ax.set_yticks(range(len(order)), [f"{c} (n={k})" for c, k in zip(order, n)])
ax.set_xlabel("예측"); ax.set_ylabel("실제")
ax.set_title(f"Test 혼동행렬 — Macro F1 {f1_score(y_test, pred, average='macro'):.4f} (행 기준 %, 괄호 = 건수)")

ax = axes[1]                                                                     # (오른쪽) 클래스별 Precision / Recall / F1
y_pos = np.arange(len(order))
for k, (vals, name, color) in enumerate([(p, "Precision", "tab:blue"), (r, "Recall", "tab:orange"), (f, "F1", "tab:green")]):
    ax.barh(y_pos + (k - 1) * 0.27, vals, height=0.27, label=name, color=color)
for i, v in enumerate(f):
    ax.text(v + 0.01, i + 0.27, f"{v:.4f}", va="center", fontsize=8)
ax.set_yticks(y_pos, order); ax.invert_yaxis(); ax.set_xlim(0, 1.12)
ax.set_title("클래스별 Precision · Recall · F1"); ax.grid(alpha=0.3, axis="x"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3)
plt.tight_layout(); plt.show()

# 주요 오류 요약
yt, yp = y_test.values, np.asarray(pred)
MINOR = ["Pastry", "Stains", "Dirtiness"]
wrong = yt != yp
unseen = ~plate_key(X_test).isin(plate_counts.keys()).values                    # Train에 없던 강판 사양
key_errors = pd.Series({
    "전체 오분류": int(wrong.sum()),
    "소수 유형 놓침 (Pastry·Stains·Dirtiness → Other_Faults·Bumps)": int((np.isin(yt, MINOR) & np.isin(yp, ["Other_Faults", "Bumps"])).sum()),
    "오경보 (Other_Faults → 소수 유형)": int(((yt == "Other_Faults") & np.isin(yp, MINOR)).sum()),
    "Bumps ↔ Other_Faults 혼동": int((((yt == "Bumps") & (yp == "Other_Faults")) | ((yt == "Other_Faults") & (yp == "Bumps"))).sum()),
    "Other_Faults ↔ Pastry 혼동": int((((yt == "Pastry") & (yp == "Other_Faults")) | ((yt == "Other_Faults") & (yp == "Pastry"))).sum()),
    "K_Scratch ↔ Z_Scratch 혼동": int((((yt == "K_Scratch") & (yp == "Z_Scratch")) | ((yt == "Z_Scratch") & (yp == "K_Scratch"))).sum())}, name="건수")
display(key_errors.to_frame())

paths = pd.Series([f"{a} → {b}" for a, b in zip(yt[wrong], yp[wrong])]).value_counts().head(6)
display(paths.rename("건수").to_frame().rename_axis("오분류 경로 (상위 6)"))

print("주요 결과")
print(f"- Test Macro F1 {f1_score(yt, yp, average='macro'):.4f} | Accuracy {(~wrong).mean():.4f} | "
      f"Macro Precision {p.mean():.4f} | Macro Recall {r.mean():.4f}")
print(f"- 가장 잘 맞힌 유형: {order[int(f.argmax())]} (F1 {f.max():.4f}) / 가장 약한 유형: {order[int(f.argmin())]} (F1 {f.min():.4f})")
print(f"- 오분류 {wrong.sum()}건 중 Bumps ↔ Other_Faults {key_errors.iloc[3]}건 ({key_errors.iloc[3] / wrong.sum():.1%})")
print(f"- Train에 없던 강판 사양의 결함 {unseen.sum()}건 중 {wrong[unseen].sum()}건 오분류 ({wrong[unseen].mean():.1%}) "
      f"vs 기존 사양 {wrong[~unseen].mean():.1%}")

In [ ]:
# 6-2. 순열 중요도 (README 8.3절): 원본 26개 열을 하나씩 뒤섞었을 때 Test Macro F1이 얼마나 떨어지는가
#      파생변수는 원본 열로 계산되므로, 원본 열을 섞으면 그 열로 만든 파생변수까지 함께 망가진다.
#      → "그 정보를 통째로 못 쓰게 되면 얼마나 손해인가"를 잰다. (모델 확정 후 해석용, 열마다 3회 반복)
def macro_f1_on(X_eval):
    Xf = add_features(X_eval, plate_counts).replace([np.inf, -np.inf], np.nan).fillna(0)   # 섞인 값으로 생길 수 있는 0 나눗셈 방지
    proba = np.mean([m.predict_proba(Xf) for m in models], axis=0)
    return f1_score(y_test, models[0].classes_[proba.argmax(axis=1)], average="macro")

base_f1 = macro_f1_on(X_test)
rng = np.random.default_rng(RS)
rows = []
for col in X_test.columns:
    drops = []
    for _ in range(3):
        X_perm = X_test.copy()
        X_perm[col] = rng.permutation(X_perm[col].to_numpy())
        drops.append(base_f1 - macro_f1_on(X_perm))
    rows.append({"변수": col, "Macro F1 감소량": np.mean(drops), "표준편차": np.std(drops)})

perm_imp = pd.DataFrame(rows).sort_values("Macro F1 감소량", ascending=False).reset_index(drop=True)
perm_imp.index += 1
print(f"기준 Test Macro F1 {base_f1:.4f}")
perm_imp.head(10).round(4)

In [ ]:
# 7. 모델 저장 — 세 모델 + 강판 조건별 결함 수(Train 기준)를 함께 저장 (예측 시 파생변수 계산에 필요)
MODEL_PATH = ROOT / "models" / "Taegyeom_final_model_trained.pkl"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
bundle = {"models": models,                      # [ExtraTrees, RandomForest, GBM]
          "plate_counts": plate_counts,          # plate_defect_count 계산용
          "feature_columns": list(add_features(X_train.head(1), plate_counts).columns),   # 학습 시 열 순서 (36개)
          "test_macro_f1": round(f1_score(y_test, pred, average="macro"), 4)}
joblib.dump(bundle, MODEL_PATH, compress=3)      # 압축 저장 (약 16MB)
print("저장:", MODEL_PATH)